# CityLearn Tutorial - Understanding the Environment and Data

This notebook follows the CityLearn tutorial to understand the environment structure and data.

## 1. Environment Setup and Imports

In [ ]:
import sys
import os
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from citylearn import CityLearnEnv
import gymnasium as gym

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

## 2. Load CityLearn Environment

We'll start with the 2023 CityLearn Challenge dataset.

In [ ]:
# Load the CityLearn environment with the 2023 dataset
schema_path = '../data/citylearn_challenge_2023_phase_1/schema.json'

try:
    env = CityLearnEnv(schema=schema_path)
    print(f"Environment loaded successfully!")
    print(f"Number of buildings: {len(env.buildings)}")
    print(f"Building names: {[building.name for building in env.buildings]}")
    print(f"Observation space: {env.observation_space}")
    print(f"Action space: {env.action_space}")
    print(f"Time steps: {env.time_steps}")
except Exception as e:
    print(f"Error loading environment: {e}")
    # Try alternative approach with direct data loading
    print("Loading data directly from CSV files...")
    from utils.data_utils import load_building_data
    building_data = load_building_data('../data')
    print(f"Loaded {len(building_data)} datasets")
    for name, df in building_data.items():
        print(f"  {name}: {df.shape}")

## 3. Explore Dataset Structure

Let's examine the data structure and buildings.

In [ ]:
# Reset environment and get initial observations
observations = env.reset()
print(f"Initial observations shape: {len(observations)}")
print(f"Building names: {[building.name for building in env.buildings]}")

# Get building information
for i, building in enumerate(env.buildings):
    print(f"\nBuilding {i+1}: {building.name}")
    print(f"  Observation space: {building.observation_space}")
    print(f"  Action space: {building.action_space}")

## 4. Data Analysis

Analyze the time series data for each building.

In [ ]:
# Extract time series data
building_data = {}

for i, building in enumerate(env.buildings):
    # Get the underlying data
    data = {
        'cooling_demand': building.energy_simulation.cooling_demand,
        'heating_demand': building.energy_simulation.heating_demand,
        'dhw_demand': building.energy_simulation.dhw_demand,
        'non_shiftable_load': building.energy_simulation.non_shiftable_load,
        'solar_generation': building.pv.get_generation(building.energy_simulation.solar_generation)
    }
    
    building_data[building.name] = pd.DataFrame(data)
    print(f"Building {building.name}: {len(building_data[building.name])} timesteps")

# Display basic statistics
for building_name, df in building_data.items():
    print(f"\n=== {building_name} Statistics ===")
    print(df.describe())

## 5. Visualize Time Series Data

Create plots to understand the temporal patterns.

In [ ]:
# Plot energy demands for all buildings
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Energy Demand Patterns - All Buildings', fontsize=16)

columns = ['cooling_demand', 'heating_demand', 'dhw_demand', 'solar_generation']
axes = axes.flatten()

for i, column in enumerate(columns):
    for building_name, df in building_data.items():
        # Plot first 1000 timesteps for visibility
        axes[i].plot(df[column][:1000], label=building_name, alpha=0.7)
    
    axes[i].set_title(f'{column.replace("_", " ").title()}')
    axes[i].set_xlabel('Time Step')
    axes[i].set_ylabel('Energy (kWh)')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Correlation Analysis

Analyze correlations between different energy components.

In [ ]:
# Create correlation matrices for each building
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Energy Components Correlation Matrix', fontsize=16)

building_names = list(building_data.keys())
axes = axes.flatten()

for i, (building_name, df) in enumerate(building_data.items()):
    if i < 4:  # Only plot first 4 buildings
        corr_matrix = df.corr()
        
        sns.heatmap(corr_matrix, 
                   annot=True, 
                   cmap='coolwarm', 
                   center=0, 
                   ax=axes[i])
        axes[i].set_title(f'{building_name}')

plt.tight_layout()
plt.show()

## 7. Data Splitting for ML

Prepare data splits for both RL and forecasting tasks.

In [ ]:
# Define split ratios
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

# Split data for each building
data_splits = {}

for building_name, df in building_data.items():
    n_samples = len(df)
    
    train_end = int(n_samples * train_ratio)
    val_end = int(n_samples * (train_ratio + val_ratio))
    
    splits = {
        'train': df.iloc[:train_end],
        'val': df.iloc[train_end:val_end],
        'test': df.iloc[val_end:]
    }
    
    data_splits[building_name] = splits
    
    print(f"{building_name}:")
    print(f"  Train: {len(splits['train'])} samples ({train_ratio*100:.0f}%)")
    print(f"  Val: {len(splits['val'])} samples ({val_ratio*100:.0f}%)")
    print(f"  Test: {len(splits['test'])} samples ({test_ratio*100:.0f}%)")
    print()

## 8. Save Processed Data

Save the processed data for use in RL and forecasting experiments.

In [ ]:
# Create data directory if it doesn't exist
os.makedirs('../data', exist_ok=True)

# Save raw building data
for building_name, df in building_data.items():
    df.to_csv(f'../data/{building_name}_raw.csv', index=False)

# Save data splits
for building_name, splits in data_splits.items():
    for split_name, split_data in splits.items():
        filename = f'../data/{building_name}_{split_name}.csv'
        split_data.to_csv(filename, index=False)

print("Data saved successfully!")
print("Files created:")
for file in os.listdir('../data'):
    if file.endswith('.csv'):
        print(f"  - {file}")

## 9. CityLearn Challenge 2023 Specific Analysis

Let's examine the data with the official CityLearn Challenge 2023 structure and targets.

In [ ]:
# Test CityLearn Challenge specific implementation
from forecasting.citylearn_challenge import CityLearnChallengeForecaster

# Create challenger instance
challenger = CityLearnChallengeForecaster(data_path='../data')

# Load Phase 1 data
try:
    challenger.load_challenge_data('phase_1')
    
    # Display dataset info
    print("\nCityLearn Challenge 2023 Dataset Analysis")
    print("=" * 50)
    
    building_names = [k for k in challenger.data.keys() if k.startswith('Building_')]
    print(f"Buildings: {len(building_names)} ({building_names})")
    
    # Show building-level targets
    if building_names:
        sample_building = challenger.data[building_names[0]]
        print(f"\nBuilding columns ({building_names[0]}):")
        for col in sample_building.columns:
            if col in challenger.building_targets:
                print(f"  TARGET: {col}")
            else:
                print(f"         {col}")
        
        print(f"\nDataset length: {len(sample_building)} timesteps")
        print(f"Prediction horizon: {challenger.prediction_horizon} hours")
        print(f"Input sequence length: {challenger.sequence_length} hours")
    
    # Show neighborhood-level data
    if 'carbon_intensity' in challenger.data:
        carbon_data = challenger.data['carbon_intensity']
        print(f"\nCarbon intensity data: {len(carbon_data)} timesteps")
        print(f"Range: {carbon_data['carbon_intensity'].min():.4f} - {carbon_data['carbon_intensity'].max():.4f} kgCO2e/kWh")
    
    print("\nOfficial CityLearn Challenge Targets:")
    print("Building-level (per building):")
    for target in challenger.building_targets:
        print(f"  - {target}")
    print("Neighborhood-level (aggregated):")
    for target in challenger.neighborhood_targets:
        print(f"  - {target}")
        
except Exception as e:
    print(f"Error: {e}")

## 10. Project Results Analysis

Load and analyze the experimental results from our implemented algorithms.

In [ ]:
# Load and analyze the actual project results
import json

# Load experimental results
try:
    with open('../results/forecasting_results.json', 'r') as f:
        forecasting_results = json.load(f)
    
    with open('../results/cross_building_results.json', 'r') as f:
        cross_building_results = json.load(f)
    
    with open('../results/neighborhood_results.json', 'r') as f:
        neighborhood_results = json.load(f)
    
    print("Project Results Analysis")
    print("=" * 40)
    
    # Analyze forecasting results
    print("\n1. FORECASTING PERFORMANCE:")
    for target, target_results in forecasting_results.items():
        print(f"\nTarget: {target}")
        for building, building_results in target_results.items():
            print(f"  {building}:")
            for model, metrics in building_results.items():
                if 'rmse' in metrics:
                    print(f"    {model}: RMSE={metrics['rmse']:.4f}, NRMSE={metrics['nrmse']:.4f}")
    
    # Analyze cross-building results  
    print("\n2. CROSS-BUILDING GENERALIZATION:")
    for train_building, train_results in cross_building_results.items():
        if isinstance(train_results, dict):
            print(f"\nTrained on {train_building}:")
            for model, model_results in train_results.items():
                if isinstance(model_results, dict):
                    print(f"  {model}:")
                    for test_building, metrics in model_results.items():
                        if 'rmse' in metrics:
                            print(f"    -> {test_building}: RMSE={metrics['rmse']:.4f}")
    
    # Analyze neighborhood results
    print("\n3. NEIGHBORHOOD AGGREGATION:")
    for target, target_results in neighborhood_results.items():
        print(f"\n{target}:")
        for model, metrics in target_results.items():
            if 'rmse' in metrics:
                print(f"  {model}: RMSE={metrics['rmse']:.4f}")
                
except Exception as e:
    print(f"Error loading results: {e}")
    print("Run the experiments first with: python run_experiments.py")

## 11. Visualization of Results

Create visualizations of the experimental results.

In [ ]:
# Create visualizations of the experimental results
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Load results table
try:
    results_df = pd.read_csv('../results/tables/detailed_results.csv')
    
    # Create performance comparison plots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('CityLearn Challenge 2023 - Model Performance Comparison', fontsize=16)
    
    # Plot 1: RMSE by Model and Building (Cooling Demand)
    cooling_data = results_df[results_df['Target'] == 'cooling_demand']
    if not cooling_data.empty:
        pivot_rmse = cooling_data.pivot(index='Building', columns='Model', values='RMSE')
        pivot_rmse.plot(kind='bar', ax=axes[0,0])
        axes[0,0].set_title('Cooling Demand - RMSE by Model')
        axes[0,0].set_ylabel('RMSE')
        axes[0,0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        axes[0,0].tick_params(axis='x', rotation=45)
    
    # Plot 2: NRMSE by Model and Building (Cooling Demand)
    if not cooling_data.empty:
        pivot_nrmse = cooling_data.pivot(index='Building', columns='Model', values='NRMSE')
        pivot_nrmse.plot(kind='bar', ax=axes[0,1])
        axes[0,1].set_title('Cooling Demand - NRMSE by Model')
        axes[0,1].set_ylabel('NRMSE')
        axes[0,1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        axes[0,1].tick_params(axis='x', rotation=45)
    
    # Plot 3: Solar Generation Performance
    solar_data = results_df[results_df['Target'] == 'solar_generation']
    if not solar_data.empty:
        pivot_solar = solar_data.pivot(index='Building', columns='Model', values='NRMSE')
        pivot_solar.plot(kind='bar', ax=axes[1,0])
        axes[1,0].set_title('Solar Generation - NRMSE by Model')
        axes[1,0].set_ylabel('NRMSE')
        axes[1,0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        axes[1,0].tick_params(axis='x', rotation=45)
    
    # Plot 4: Overall Model Ranking
    model_avg = results_df.groupby('Model')['NRMSE'].mean().sort_values()
    model_avg.plot(kind='bar', ax=axes[1,1])
    axes[1,1].set_title('Average NRMSE Across All Targets')
    axes[1,1].set_ylabel('Average NRMSE')
    axes[1,1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\nMODEL PERFORMANCE SUMMARY:")
    print("=" * 40)
    print("\nAverage NRMSE by Model:")
    for model, nrmse in model_avg.items():
        print(f"  {model}: {nrmse:.4f}")
        
    print("\nBest model per target:")
    for target in results_df['Target'].unique():
        target_data = results_df[results_df['Target'] == target]
        best_model = target_data.loc[target_data['NRMSE'].idxmin()]
        print(f"  {target}: {best_model['Model']} (NRMSE: {best_model['NRMSE']:.4f})")
        
except Exception as e:
    print(f"Error creating visualizations: {e}")
    print("Make sure to run the experiments first: python run_experiments.py")

## 12. Conclusions

This tutorial demonstrated the complete CityLearn Challenge 2023 implementation.

### Project Components:
- **Reinforcement Learning**: Q-Learning and SAC algorithms
- **Time Series Forecasting**: Multiple ML models (Linear, RF, Gaussian, LSTM, ANN, Transformer)
- **Advanced Analysis**: Cross-building generalization and neighborhood aggregation
- **Evaluation**: 80% training split with normalized RMSE metrics

### Key Results:
- Linear Regression shows robust performance across buildings
- Random Forest excels at solar generation forecasting  
- Cross-building transfer shows moderate generalization capability
- Neighborhood aggregation provides stable forecasting targets

The implementation successfully addresses all CityLearn Challenge 2023 requirements with comprehensive evaluation and analysis.

In [ ]:
# Project completion summary
print("Tutorial completed successfully!")
print("All project components implemented and tested.")
print("Results available in ../results/ directory.")
print("Repository: https://github.com/leleadami/citylearn-challenge-2023")